# Tarea 7.2

Calcular $I = \int_0^4 \frac{dx}{\sqrt{x^2+16}}$ con trapecio y Simpson compuestos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f = lambda x: 1 / np.sqrt(x**2 + 16)
a, b = 0, 4

# la primitiva es arcsinh(x/4), entonces I = arcsinh(1)
I_exacto = np.arcsinh(1)
print("Valor exacto:", I_exacto)

Trapecio compuesto:

$$T_n = \frac{h}{2}\left[f(x_0) + 2\sum_{i=1}^{n-1} f(x_i) + f(x_n)\right]$$

In [ ]:
def trapecio(f, a, b, n):
    x = np.linspace(a, b, n+1)
    y = f(x)
    h = (b-a)/n
    return h*(y[0]/2 + np.sum(y[1:-1]) + y[-1]/2)

Simpson compuesto (n tiene que ser par):

$$S_n = \frac{h}{3}\left[f(x_0) + 4\sum_{\text{impares}} f(x_i) + 2\sum_{\text{pares}} f(x_i) + f(x_n)\right]$$

In [ ]:
def simpson(f, a, b, n):
    if n % 2 != 0:
        raise ValueError("n tiene que ser par")
    x = np.linspace(a, b, n+1)
    y = f(x)
    h = (b-a)/n
    return h/3*(y[0] + 4*np.sum(y[1:-1:2]) + 2*np.sum(y[2:-1:2]) + y[-1])

Ahora calculamos el error para varios $n$ (todos pares):

In [ ]:
ns = 2**np.arange(1, 9)   # 2, 4, 8, ..., 256
hs, eT, eS = [], [], []

for n in ns:
    h = (b-a)/n
    hs.append(h)
    eT.append(abs(trapecio(f, a, b, n) - I_exacto))
    eS.append(abs(simpson(f, a, b, n) - I_exacto))

hs, eT, eS = np.array(hs), np.array(eT), np.array(eS)

print(f"{'n':>5} {'h':>10} {'err trap':>12} {'err Simp':>12}")
for n, h, t, s in zip(ns, hs, eT, eS):
    print(f"{n:>5} {h:>10.5f} {t:>12.3e} {s:>12.3e}")

Fórmulas de error.. Las derivadas son $f'(x) = -x(x^2+16)^{-3/2}$ y $f'''(x) = 6x(24-x^2)(x^2+16)^{-7/2}$. Como en $x=0$ se anulan, sólo queda el término en $b=4$.

In [ ]:
fp   = lambda x: -x*(x**2+16)**(-1.5)
fppp = lambda x: 6*x*(24-x**2)*(x**2+16)**(-3.5)

teoT = np.abs(hs**2/12 * (fp(b) - fp(a)))
teoS = np.abs(hs**4/180 * (fppp(b) - fppp(a)))

Gráfica del error contra $h$:

In [ ]:
plt.figure(figsize=(8,6))
plt.loglog(hs, eT, 'o-', label='Trapecio (numérico)')
plt.loglog(hs, eS, 's-', label='Simpson (numérico)')
plt.loglog(hs, teoT, '--', label='Teórico trapecio')
plt.loglog(hs, teoS, '--', label='Teórico Simpson')
plt.gca().invert_xaxis()
plt.xlabel('h')
plt.ylabel('error absoluto')
plt.title('Error vs h')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.show()

Pendientes en log-log, deberían dar cerca de 2 y 4:

In [ ]:
pT = np.polyfit(np.log(hs), np.log(eT), 1)[0]
pS = np.polyfit(np.log(hs), np.log(eS), 1)[0]
print("pendiente trapecio:", round(pT, 3))
print("pendiente Simpson:", round(pS, 3))

Finalmente: Simpson ($O(h^4)$) es mucho más preciso que el metodo del ttrapecio ($O(h^2)$) para el mismo $h$. Para $h$ muy chico el error de Simpson ya llega al ruido de redondeo y la curva deja de bajar. Para $h$ grande las fórmulas teóricas no coinciden tan bien porque todavía pesan los términos de orden superior.